In [51]:
import os
import yfinance as yf
import pandas as pd
from scipy.optimize import brentq
from scipy.stats import norm
from datetime import datetime, timedelta,date
import requests
import json
from dotenv import load_dotenv
from polygon import RESTClient
from datetime import date as date_type, timedelta
import pandas as pd
import re
import numpy as np

load_dotenv()
POLYGON_API_KEY = os.getenv("Polygon_API_Key")
if not POLYGON_API_KEY:
    raise ValueError("Set Polygon_API_Key in your .env file")


## Historical Price of Underlying Asset

In [59]:
symbol = "QQQ"
ticker = yf.Ticker(symbol)
history = ticker.history(start="2023-12-01", actions=True, auto_adjust=False)
underlying_df = pd.DataFrame(history)
underlying_df.reset_index(inplace=True)
underlying_df["Date"] = underlying_df["Date"].dt.date
if "Dividends" not in underlying_df.columns:
    underlying_df["Dividends"] = 0.0
else:
    underlying_df["Dividends"] = underlying_df["Dividends"].fillna(0.0)
underlying_df["Return"] = underlying_df["Close"].pct_change()
underlying_df["RV"] = underlying_df["Return"].rolling(window=20).std() * np.sqrt(252)
underlying_df = underlying_df.set_index("Date").sort_index()


## First calendar day of each month (option selection date)

In [60]:
start_date = date(2024, 5, 1)
end_date = date(2026, 3, 31)

month_starts = []
y, m = start_date.year, start_date.month
while True:
    first = date(y, m, 1)
    if first > end_date:
        break
    if first >= start_date:
        month_starts.append(first)
    if m == 12:
        y, m = y + 1, 1
    else:
        m += 1

## Underlying series
yfinance history is kept **in memory** as `underlying_df` (Date index). Only `DataSet/{symbol}.csv` is written to disk.

## 1-month Treasury yield as risk-free rate over time (Polygon)
Polygon does not provide 1-month OIS; we use **1-month U.S. Treasury yield** from Polygon's Fed API as the risk-free rate for each date.

In [61]:
# Fetch 1-month Treasury yield over time (Polygon) as risk-free rate
# Polygon does not offer 1-month OIS; we use 1-month Treasury yield (closest short-term risk-free proxy).
client = RESTClient(POLYGON_API_KEY)
from_date = min(month_starts).strftime("%Y-%m-%d")
to_date = date.today().strftime("%Y-%m-%d")
# yields_raw: list of TreasuryYield(date='YYYY-MM-DD', yield_1_month=5.48, yield_3_month=..., ...); yield_1_month in %.
yields_raw = list(client.list_treasury_yields(date_gte=from_date, date_lte=to_date, limit=50000, sort="date", order="asc"))
r_fallback = 0.0365
# print(yields_raw[0].date,yields_raw[0].yield_1_month)
if not yields_raw:
    r_by_date = pd.Series(r_fallback, index=pd.date_range(from_date, to_date, freq="D"))
    print("No Treasury yields from Polygon; using fallback rate for all dates.")
else:
    # Loop over yields_raw: record date and yield_1_month for each i; yield in % -> decimal r
    dates_list = []
    rates_list = []
    for i in range(len(yields_raw)):
        d = yields_raw[i].date
        y = yields_raw[i].yield_1_month
        dates_list.append(pd.to_datetime(d))
        rates_list.append((float(y) / 100.0) if y is not None else np.nan)
    r_by_date = pd.Series(rates_list, index=dates_list)
    r_by_date = r_by_date.sort_index()
    # Missing calendar days: use rate from previous day (ffill)
    full_range = pd.date_range(from_date, to_date, freq="D")
    r_by_date = r_by_date.reindex(full_range).ffill().fillna(r_fallback)
    r_by_date = r_by_date.astype(np.float64)
    print(f"Loaded 1-month Treasury yields from {r_by_date.index.min()} to {r_by_date.index.max()} (n={len(r_by_date)})")

Loaded 1-month Treasury yields from 2024-05-01 00:00:00 to 2026-04-12 00:00:00 (n=712)


In [62]:
# underlying_df is built from yfinance in the cell above

## ATM options, **first Friday** of next month (primary)

If that expiration has no chain (e.g. holiday like **2025-07-04**), the notebook falls back to **second Friday**, then any other Polygon expiration in **days 1–7** of next month (by distance from the first Friday). Strikes use the nearest common call/put strike to `s0`; if none align, it uses the best call and best put separately (with a warning).

In [63]:
def atm_option(symbol, s0, date):
    client = RESTClient(POLYGON_API_KEY)

    date_obj = date
    date_str = date_obj.strftime("%Y-%m-%d")

    if date_obj.month == 12:
        next_month_first = date_obj.replace(year=date_obj.year + 1, month=1, day=1)
    else:
        next_month_first = date_obj.replace(month=date_obj.month + 1, day=1)

    ny, nm = next_month_first.year, next_month_first.month
    days_to_friday = (4 - next_month_first.weekday()) % 7
    first_friday = next_month_first + timedelta(days=days_to_friday)
    second_friday = first_friday + timedelta(weeks=1)
    exp_first = first_friday.strftime("%Y-%m-%d")
    exp_second = second_friday.strftime("%Y-%m-%d")

    # Polygon caps ListOptionsContracts limit at 1000; client paginates beyond that.
    contracts = list(
        client.list_options_contracts(underlying_ticker=symbol, as_of=date_str, limit=1000)
    )
    if not contracts:
        print(f"No options contracts found as of {date_str}.")
        return None, None

    def exp_in_first_week(exp_s: str) -> bool:
        ed = datetime.strptime(exp_s, "%Y-%m-%d").date()
        return ed.year == ny and ed.month == nm and 1 <= ed.day <= 7

    extra_exps = sorted(
        {c.expiration_date for c in contracts if c.expiration_date and exp_in_first_week(c.expiration_date)},
        key=lambda s: abs((datetime.strptime(s, "%Y-%m-%d").date() - first_friday).days),
    )
    expiry_order = []
    for e in [exp_first, exp_second] + extra_exps:
        if e not in expiry_order:
            expiry_order.append(e)

    def pick_pair(calls, puts, exp_used: str):
        common = sorted(
            {c.strike_price for c in calls} & {p.strike_price for p in puts},
            key=lambda k: abs(k - s0),
        )
        if common:
            k = common[0]
            atm_call = next(c for c in calls if c.strike_price == k)
            atm_put = next(p for p in puts if p.strike_price == k)
            if exp_used != exp_first:
                print(
                    f"ATM expiry fallback: using {exp_used} (no chain on primary {exp_first} as of {date_str})."
                )
            return atm_call.ticker, atm_put.ticker
        atm_call = min(calls, key=lambda c: abs(c.strike_price - s0))
        atm_put = min(puts, key=lambda p: abs(p.strike_price - s0))
        if exp_used != exp_first:
            print(
                f"ATM expiry fallback: using {exp_used} (no chain on primary {exp_first} as of {date_str})."
            )
        print(
            "Warning: no strike with both call and put; using best call / best put (strikes may differ)."
        )
        return atm_call.ticker, atm_put.ticker

    for exp in expiry_order:
        calls = [c for c in contracts if c.expiration_date == exp and c.contract_type == "call"]
        puts = [c for c in contracts if c.expiration_date == exp and c.contract_type == "put"]
        if not calls or not puts:
            continue
        return pick_pair(calls, puts, exp)

    print(
        f"No ATM pair found as of {date_str}; tried expiries (first week next month): {expiry_order}"
    )
    return None, None

## Option daily bars + implied vol (fetched over each calendar month; panel rows are **weekdays only**)

In [64]:
# Black-Scholes formula for calls and puts
def bs_call_price(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)


def bs_put_price(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)


IV_FALLBACK = 1.0


def implied_vol_call(market_price, S, K, T, r):
    def obj_func(sigma):
        return bs_call_price(S, K, T, r, sigma) - market_price

    try:
        return brentq(obj_func, 1e-6, 5)
    except ValueError:
        return IV_FALLBACK


def implied_vol_put(market_price, S, K, T, r):
    def obj_func(sigma):
        return bs_put_price(S, K, T, r, sigma) - market_price

    try:
        return brentq(obj_func, 1e-6, 5)
    except ValueError:
        return IV_FALLBACK


def option_bars_with_iv(
    option_ticker: str,
    d_start: date,
    d_end: date,
    r_by_date: pd.Series,
    underlying_df: pd.DataFrame,
    r_fallback: float,
) -> pd.DataFrame:
    """Daily option close + implied vol between d_start and d_end (inclusive)."""
    client = RESTClient(POLYGON_API_KEY)
    from_str = d_start.strftime("%Y-%m-%d")
    to_str = d_end.strftime("%Y-%m-%d")
    aggs = client.get_aggs(
        ticker=option_ticker,
        multiplier=1,
        timespan="day",
        from_=from_str,
        to=to_str,
        limit=50000,
    )
    if not aggs:
        return pd.DataFrame(columns=["date", "close", "imp_vol"])

    rows = []
    for agg in aggs:
        rows.append(
            {
                "date": pd.to_datetime(agg.timestamp, unit="ms").date(),
                "close": agg.close,
                "imp_vol": getattr(agg, "implied_volatility", None) or getattr(agg, "iv", None),
            }
        )
    df = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)

    symbol_part = option_ticker.split(":")[1]
    match = re.search(r"^[A-Z]+(\d{6})([CP])(\d{8})$", symbol_part)
    if not match:
        raise ValueError(
            f"Could not parse option symbol: {option_ticker}. Expected format like 'AAPL240209C00185000'."
        )

    expiry_str = match.group(1)
    option_type = match.group(2)
    strike_str = match.group(3)
    if len(expiry_str) == 6:
        expiry_str = "20" + expiry_str
    expiry = pd.to_datetime(expiry_str, format="%Y%m%d").date()
    K = int(strike_str) / 1000.0

    df["ttm"] = df["date"].apply(lambda x: max((expiry - x).days / 365.25, 1e-9))
    if isinstance(r_by_date, pd.Series):
        df["r"] = (
            r_by_date.reindex(pd.to_datetime(df["date"])).ffill().bfill().fillna(r_fallback).values
        )
    else:
        df["r"] = float(r_by_date)

    spot_df = underlying_df[["Close"]].reset_index()
    spot_df.columns = ["date", "S"]
    spot_df["date"] = pd.to_datetime(spot_df["date"]).dt.date
    df = df.merge(spot_df, on="date", how="left")
    last_S = float(underlying_df["Close"].iloc[-1]) if len(underlying_df) else np.nan
    df["S"] = df["S"].ffill().bfill().fillna(last_S)

    df["imp_vol"] = pd.to_numeric(df["imp_vol"], errors="coerce")
    need_iv = df["imp_vol"].isna() | ~np.isfinite(df["imp_vol"])
    if need_iv.any():
        if option_type == "C":
            df.loc[need_iv, "imp_vol"] = df.loc[need_iv].apply(
                lambda row: implied_vol_call(row["close"], row["S"], K, row["ttm"], row["r"]),
                axis=1,
            )
        else:
            df.loc[need_iv, "imp_vol"] = df.loc[need_iv].apply(
                lambda row: implied_vol_put(row["close"], row["S"], K, row["ttm"], row["r"]),
                axis=1,
            )
    df["imp_vol"] = df["imp_vol"].astype(float)
    return df.drop(columns=["ttm", "r", "S"])

## One CSV per underlying: **equity session** rows per month

Start from pandas **weekdays** in the month, then **drop rows where `Stock_Close` is NaN** (NYSE holidays on a weekday). `Force_Close` is `True` on the **last remaining row** of that month (last trading session in the file).

In [65]:
import calendar


def month_last(mfirst: date) -> date:
    return date(mfirst.year, mfirst.month, calendar.monthrange(mfirst.year, mfirst.month)[1])


def spot_asof(underlying: pd.DataFrame, d: date) -> float:
    sub = underlying.loc[underlying.index <= d]
    if sub.empty:
        return np.nan
    return float(sub["Close"].iloc[-1])


all_rows = []
for month_first in month_starts:
    month_end = month_last(month_first)
    s0 = spot_asof(underlying_df, month_first)
    if not np.isfinite(s0):
        print(f"Skip {month_first}: no underlying spot")
        continue
    atm_call, atm_put = atm_option(symbol, s0, month_first)
    if atm_call is None or atm_put is None:
        print(f"Skip {month_first}: no ATM pair")
        continue

    call_df = option_bars_with_iv(
        atm_call, month_first, month_end, r_by_date, underlying_df, r_fallback
    )
    put_df = option_bars_with_iv(
        atm_put, month_first, month_end, r_by_date, underlying_df, r_fallback
    )

    bdays = pd.bdate_range(month_first, month_end, freq="B")
    panel = pd.DataFrame({"Date": bdays.date})

    u = underlying_df.reset_index().rename(
        columns={"Close": "Stock_Close", "Dividends": "Stock_Dividends"}
    )
    panel = panel.merge(
        u[["Date", "Stock_Close", "Stock_Dividends", "RV"]], on="Date", how="left"
    )
    # Drop weekdays that are market holidays (no yfinance print)
    panel = panel.loc[panel["Stock_Close"].notna()].copy()
    if panel.empty:
        print(f"Skip {month_first}: no rows with Stock_Close (all holidays?)")
        continue

    ts = pd.to_datetime(panel["Date"])
    panel["r"] = r_by_date.reindex(ts).ffill().bfill().fillna(r_fallback).values.astype(float)

    call_df = call_df.rename(
        columns={"date": "Date", "close": "Call_Close", "imp_vol": "Call_imp_vol"}
    )
    put_df = put_df.rename(
        columns={"date": "Date", "close": "Put_Close", "imp_vol": "Put_imp_vol"}
    )
    panel = panel.merge(call_df[["Date", "Call_Close", "Call_imp_vol"]], on="Date", how="left")
    panel = panel.merge(put_df[["Date", "Put_Close", "Put_imp_vol"]], on="Date", how="left")
    panel["Call_Sym"] = atm_call
    panel["Put_Sym"] = atm_put

    panel["Force_Close"] = panel["Date"] == panel["Date"].max()

    out_cols = [
        "Date",
        "Stock_Close",
        "Stock_Dividends",
        "r",
        "RV",
        "Call_Close",
        "Call_Sym",
        "Put_Close",
        "Put_Sym",
        "Call_imp_vol",
        "Put_imp_vol",
        "Force_Close",
    ]
    all_rows.append(panel[out_cols])

if not all_rows:
    raise RuntimeError("No monthly panels built; check date range and Polygon responses.")
full = pd.concat(all_rows, ignore_index=True)
out_path = f"DataSet/{symbol}.csv"
full.to_csv(out_path, index=False)
print(f"Wrote {len(full)} rows to {out_path}")

ATM expiry fallback: using 2025-07-11 (no chain on primary 2025-07-04 as of 2025-06-01).
ATM expiry fallback: using 2026-04-10 (no chain on primary 2026-04-03 as of 2026-03-01).
Wrote 480 rows to DataSet/QQQ.csv
